# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Follows checklist.md.

Notes on structure: there is no date column and `device_serial` is unique per row (no repeated entities), so a random stratified split is the appropriate test-set design. `device_serial` is an identifier and is dropped. All other columns (channel, device age, price, daily hours at registration) are recorded at sale/registration, so they are known at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, df.device_serial.is_unique, df.isna().sum().sum())
print(df.claim_within_year.mean())
df.describe()

## 1. Test set (set aside before any modelling)
20% stratified random hold-out; no time column and no repeated entities.

In [ ]:
FEATURES=['channel','device_age_years','price_eur','daily_hours']
TARGET='claim_within_year'
train,test=train_test_split(df,test_size=0.2,stratify=df[TARGET],random_state=SEED)
Xtr,ytr=train[FEATURES],train[TARGET]
Xte,yte=test[FEATURES],test[TARGET]
print(len(train),len(test),ytr.mean(),yte.mean())

## 2. Candidate models (named before scoring)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: logistic regression, scaled numerics + one-hot channel; grid over C in {0.01, 0.1, 1, 10}.
3. `hgb`: histogram gradient boosting; grid over learning_rate {0.03, 0.1}, max_depth {2, 3}, fixed 200 iterations.

Selection metric: CV average precision (PR-AUC), since claims are ~12.6% of rows. Decision threshold: maximises F1 on out-of-fold predictions. All preprocessing sits inside the Pipeline, so it is refit in each fold. Cross-validation is 5-fold stratified on training rows only (rows are unique entities).

In [ ]:
num=['device_age_years','price_eur','daily_hours']
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),['channel'])])
hgb_pre=ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),['channel'])],remainder='passthrough')
cands={
 'baseline':(Pipeline([('m',DummyClassifier(strategy='prior'))]),{}),
 'logreg':(Pipeline([('p',pre),('m',LogisticRegression(max_iter=1000))]),{'m__C':[0.01,0.1,1,10]}),
 'hgb':(Pipeline([('p',hgb_pre),('m',HistGradientBoostingClassifier(max_iter=200,random_state=SEED))]),
        {'m__learning_rate':[0.03,0.1],'m__max_depth':[2,3]}),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
fits={}
for name,(pipe,grid) in cands.items():
    gs=GridSearchCV(pipe,grid or {},scoring='average_precision',cv=cv).fit(Xtr,ytr)
    fits[name]=gs
    print(f"{name:9s} CV AP={gs.best_score_:.4f} params={gs.best_params_}")

In [ ]:
# Out-of-fold ROC-AUC for reference (training rows only)
for name,gs in fits.items():
    oof=cross_val_predict(gs.best_estimator_,Xtr,ytr,cv=cv,method='predict_proba')[:,1]
    print(name,'OOF ROC-AUC',round(roc_auc_score(ytr,oof),4),'OOF AP',round(average_precision_score(ytr,oof),4))

## 3. Choose model and threshold on CV only

In [ ]:
best=max((n for n in fits if n!='baseline'),key=lambda n:fits[n].best_score_)
oof=cross_val_predict(fits[best].best_estimator_,Xtr,ytr,cv=cv,method='predict_proba')[:,1]
ths=np.linspace(0.02,0.8,157)
f1s=[f1_score(ytr,oof>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,fits[best].best_params_,'threshold',round(thr,3),'OOF F1',round(max(f1s),4))

**Decision (fixed before test scoring):** the model is the best non-baseline candidate by CV average precision shown above, with its CV-selected hyperparameters and the F1-maximising out-of-fold threshold. If its CV AP is not clearly above the baseline's (prevalence ≈ 0.126), the conclusion is that the features do not support useful prediction.

## 4. Final fit on all training rows; single test-set score

In [ ]:
final=fits[best].best_estimator_.fit(Xtr,ytr)
p=final.predict_proba(Xte)[:,1]
yh=(p>=thr).astype(int)
print(f"TEST ROC-AUC={roc_auc_score(yte,p):.4f}  AP={average_precision_score(yte,p):.4f} (prevalence {yte.mean():.4f})")
print(f"at threshold {thr:.3f}: precision={precision_score(yte,yh):.3f} recall={recall_score(yte,yh):.3f} F1={f1_score(yte,yh):.3f}")

## Conclusion
See the test-set output above, scored once. Read it against the baseline AP (= prevalence) and the CV estimates.